In [4]:
from Bio import Entrez 
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

In [2]:

#1.1 скачивание файлов
def download_records(term, n, out_file):
    Entrez.email = 'sk173619@gmail.com'
    handle = Entrez.esearch(db="nucleotide", term=term, retmax=n)
    ids = Entrez.read(handle)["IdList"]
    handle.close()
    try:
        records = []
        for id_ in ids:
            gb = Entrez.efetch(db="nucleotide", id=id_, rettype="gb", retmode="text")
            rec = SeqIO.read(gb, "genbank")
            gb.close()
            records.append(rec)
    except Exception as e:
        print(f"ошибка скачивания: {e}")
        return

    SeqIO.write(records, out_file, "genbank")
    print(f"Скачано {len(records)} записей в {out_file}")

In [3]:
#1.2объединение файлов
def unite_files(files, out_file):
    all_records = []
    for f in files:
        all_records.extend(list(SeqIO.parse(f, "genbank")))
    SeqIO.write(all_records, out_file, "genbank")
    print(f"Объединено {len(all_records)} записей в {out_file}")

In [6]:
#1.3 проверка количества cds
def count_cds(file):
    try:
        cds_count = 0
        for record in SeqIO.parse(file, "genbank"):
            for feature in record.features:
                if feature.type == "CDS":
                    cds_count += 1
        return cds_count
    except Exception as e:
        print(f"ошибка счета CDS: {e}")
        return

In [7]:
#2.сортировка по gc 
def sort_gc(input_file, output_file):
    try:
        records = list(SeqIO.parse(input_file, "genbank"))
        records.sort(key=lambda r: gc_fraction(r.seq))
    except Exception as e:
        print(f"ошибка сортировки по CDS: {e}")
        return

    with open(output_file, "w") as out:
        for r in records:
            gc = gc_fraction(r.seq)
            out.write(f"{r.id}: {r.description}, GC = {gc}\n")

In [8]:

#3. название, место, белок
def extract_info(genbank_file, out_file):
    try:
        with open(out_file, "w") as out:
            for record in SeqIO.parse(genbank_file, "genbank"):
                organism = record.annotations.get("organism", "Unknown organism")
                description = record.description
                
                for feature in record.features:
                    if feature.type == "CDS":
                        
                        location = feature.location
                        protein = feature.qualifiers.get("translation")
                        out.write(f">{record.id}\n")
                        out.write(f"Organism: {organism}\n")
                        out.write(f"Definition: {description}\n")
                        out.write(f"CDS location: {location}\n")
                        out.write(f"Protein:\n{protein}\n\n")
    except Exception as e:
        print(f"Проверьте файл. ошибка: {e}")
        return

In [9]:
#данные варианта
#1
download_records('"Prunus persica"[Organism] complete cds', 5, "prunus.gb")
download_records('"Arabidopsis thaliana"[Organism] complete cds', 5, "arabidopsis.gb")
unite_files(["prunus.gb", "arabidopsis.gb"], "all_species.gb")

cds= count_cds("all_species.gb")
if cds >= 10:
    print(f"Условие соблюдается (всего {cds} CDS)")
else:
    print(f"CDS меньше 10 {cds}")

Скачано 5 записей в prunus.gb
Скачано 5 записей в arabidopsis.gb
Объединено 10 записей в all_species.gb
Условие соблюдается (всего 10 CDS)


In [10]:
#2
sort_gc('all_species.gb','all_speciesfinal.gb')
with open ('all_speciesfinal.gb', 'r') as f:
    ff = f.read()
print(ff)


PX441354.1: Arabidopsis thaliana 4-coumarate-CoA ligase mRNA, complete cds, GC = 0.34816132858837484
PX505713.1: Arabidopsis thaliana xylan 2-O-xylosyltransferase 3 (XYXT3) mRNA, complete cds, GC = 0.3983105912930474
PX505711.1: Arabidopsis thaliana xylan 2-O-xylosyltransferase 1 (XYXT1) mRNA, complete cds, GC = 0.4176548089591568
PX505712.1: Arabidopsis thaliana xylan 2-O-xylosyltransferase 2 (XYXT2) mRNA, complete cds, GC = 0.42350623768877216
PX776815.1: Prunus persica var. persica ACA10 mRNA, complete cds, GC = 0.425552353506244
PX776816.1: Prunus persica var. persica CAMTA3 mRNA, complete cds, GC = 0.42962308598351
PX776813.1: Prunus persica var. persica ACA9 mRNA, complete cds, GC = 0.43756145526057033
PX776817.1: Prunus persica var. persica PP2C8 mRNA, complete cds, GC = 0.4380508035251426
PX776814.1: Prunus persica var. persica ACA14 mRNA, complete cds, GC = 0.43880246551218083
PX505710.1: Arabidopsis thaliana xylan 2-O-arabinofuranosyltransferase 1 (X2AT1) mRNA, complete cds, 

In [11]:
#3
extract_info('all_species.gb', 'all_speciesFINAL3.gb')
with open ('all_speciesFINAL3.gb', 'r') as f:
    ff = f.read()
print(ff)

>PX776817.1
Organism: Prunus persica var. persica
Definition: Prunus persica var. persica PP2C8 mRNA, complete cds
CDS location: [759:1809](+)
Protein:
['MKWELVDMGSLNGTLLNSQPINNPDSGSRHWGKPMELASGDVITLGTTSKVFVHISSETESQIPFGLGVASDPMALRRGGKKLPMEDVCYYQWPLPGIDQFGLFGICDGHGGAEAARSASKLLPEMVANILSDSLKRERVLSLCDASDVLKDAFFQTEAGMNHHYEGCTATLLLVWTDGKDNFFAQCANLGDSACVMNVDGKLIKMTEDHRISSYGERLRIQETGQPLKEGETRLCGLNLARMLGDKFLKQQDSRFSSEPYISQVVQIDQASRAFALMASDGLWDVISVKKAIQLVLQTRERYSKDGENLAEKTANVLLSEARTVRTKDNTSIIFLDFDTSRISCKGES']

>PX776816.1
Organism: Prunus persica var. persica
Definition: Prunus persica var. persica CAMTA3 mRNA, complete cds
CDS location: [0:3396](+)
Protein:
['MYEIQVGSGCGISVTTTRIEFMADTKRYGLGNQLDIAQILLEAKHRWLRPAEICEILRNYKKFHISSEPASMPPGGSLFLFDRKVLRYFRKDGHNWRKKKDGKTVKEAHERLKAGSVDVLHCYYAHGEENENFQRRSYWMLEEDLQHIVLVHYREVKGNRTNFNHTKGTEEAVPYSHETEEIALNSEMENSVSSSFNPNTFQMRSQATDTTSLSSAQASEFEDAESAYDHQASSRLQPFLELLQPKAEKINAGFSDAFYPMSFSNNYQEKLSAIPGVNFGSLTQAYKREDGNDAGVNYEPTKNLNSSLWEAALENSATGFQSLSFQPSFSATHSDTMGII